In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.metrics import roc_curve
from sklearn.preprocessing import normalize
from tqdm import tqdm

device = "cpu"  # use cuda only if you really have it
torch.manual_seed(42)
np.random.seed(42)


In [4]:
X = np.load("../outputs/xvectors/xvectors.npy")      
utterances = np.load("../outputs/xvectors/utterances.npy")  # list of paths

print("X shape:", X.shape)
print("Utterances:", len(utterances))
X = normalize(X)
X = torch.tensor(X, dtype=torch.float32).to(device)



X shape: (4874, 512)
Utterances: 4874


In [5]:
speakers = [u.split("/")[0] for u in utterances]

spk2id = {s: i for i, s in enumerate(sorted(set(speakers)))}
y = torch.tensor([spk2id[s] for s in speakers], dtype=torch.long)
y = y.to("cpu")
print("Number of speakers:", len(spk2id))


Number of speakers: 40


In [6]:
protocol = "../data/voxceleb1/test/veri_test2.txt"

trials = []
labels = []

with open(protocol) as f:
    for line in f:
        lab, u1, u2 = line.strip().split()
        trials.append((u1, u2))
        labels.append(int(lab))

labels = np.array(labels)
print("Trials:", len(trials))
utt2idx = {u: i for i, u in enumerate(utterances)}
X = torch.tensor(X, dtype=torch.float)
device = "cpu"
X = X.to(device)

Trials: 37611


/var/folders/h1/nbtwrd896nb0khgr0gm32yww0000gn/T/ipykernel_45632/2536277572.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [16]:
def edge_contrastive_loss_fast(Z, edge_index, margin=0.3):
    zi = Z[edge_index[0]]
    zj = Z[edge_index[1]]

    pos_sim = F.cosine_similarity(zi, zj)

    # one random negative per edge
    neg_idx = torch.randint(0, Z.size(0), (zi.size(0),), device=Z.device)
    zk = Z[neg_idx]

    neg_sim = F.cosine_similarity(zi, zk)

    loss_pos = (1 - pos_sim).mean()
    loss_neg = F.relu(neg_sim - margin).mean()

    return loss_pos + loss_neg


In [17]:
X_np = X.cpu().numpy()
sim_matrix = sim = X_np @ X_np.T

def build_threshold_graph(knn_value):
    K = knn_value
    N = sim_matrix.shape[0]
    edge_index = []

    for i in range(sim_matrix.shape[0]):
        knn = np.argsort(sim_matrix[i])[::-1][1:K+1]
        for j in knn:
          edge_index.append([i, j])

    edge_index = torch.tensor(edge_index, dtype=torch.long).T
    class GAT(torch.nn.Module):
     def __init__(self, in_dim=512, hidden=128, out_dim=512, heads=1):
        super().__init__()
        
        self.gat1 = GATConv(
            in_dim,
            hidden,
            heads=heads,
            concat=False,
            dropout=0.3
        )
        
        self.gat2 = GATConv(
            hidden,
            out_dim,
            heads=1,
            concat=False,
            dropout=0.3
        )

     def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = self.gat2(x, edge_index)
        return F.normalize(x, dim=1)

    device = "cpu"

    model = GAT(in_dim=X.shape[1], out_dim=X.shape[1]).to(device)
    edge_index = edge_index.to(device)


    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    epochs = 15
    model.train()
    for epoch in range(epochs):
     optimizer.zero_grad()

     Z = model(X, edge_index)
     Z = F.normalize(Z, dim=1)
     loss = edge_contrastive_loss_fast(Z, edge_index)
     loss.backward()
     optimizer.step()
     
    model.eval()
    with torch.no_grad():
      Z = model(X, edge_index).cpu().numpy()

    scores = np.zeros(len(trials), dtype=np.float32)

    for k, (u1, u2) in enumerate(trials):
      i = utt2idx[u1]
      j = utt2idx[u2]
      scores[k] = np.dot(
      Z[i],
      Z[j]
      )
    
    eer, min_dcf = compute_eer_and_min_dcf(scores, labels) # type: ignore

    return eer, min_dcf   





/var/folders/h1/nbtwrd896nb0khgr0gm32yww0000gn/T/ipykernel_45632/2158009943.py:2: RuntimeWarning: divide by zero encountered in matmul
  sim_matrix = sim = X_np @ X_np.T
/var/folders/h1/nbtwrd896nb0khgr0gm32yww0000gn/T/ipykernel_45632/2158009943.py:2: RuntimeWarning: overflow encountered in matmul
  sim_matrix = sim = X_np @ X_np.T
/var/folders/h1/nbtwrd896nb0khgr0gm32yww0000gn/T/ipykernel_45632/2158009943.py:2: RuntimeWarning: invalid value encountered in matmul
  sim_matrix = sim = X_np @ X_np.T


In [9]:
def compute_eer_and_min_dcf(scores, labels):
   p_target=0.01
   c_miss=1.0
   c_fa=1.0

   fpr, tpr , _ = roc_curve(labels, scores, pos_label=1)

   fnr = 1 - tpr 

   dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
   min_dcf = np.min(dcf)

   eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
   return eer, min_dcf

In [10]:
def run_k_times(K, runs=5):
    eers, dcfs = [], []
    for _ in range(runs):
        eer, dcf = build_threshold_graph(K)
        eers.append(eer)
        dcfs.append(dcf)
    return np.mean(eers), np.mean(dcfs)


In [18]:
threshold = [40]
results_err = {}
results_dcf = {}

for t in threshold:
    err, min_dcf = run_k_times(t)
    results_err[t] = err
    results_dcf[t] = min_dcf
    print(f"GAT-refined EER for KNN {t}: {err * 100:.2f}%")
    print(f"GAT-refined MinDCF for KNN {t}: {min_dcf:.5f}")

GAT-refined EER for KNN 40: 6.94%
GAT-refined MinDCF for KNN 40: 0.00557
